# 03 - Goal Rescue: Does a Goal Paper Over a Bad Performance?

A goal is the most visible event in a match and carries the largest bonus. The risk this notebook tests is that a player who played
poorly but scored lands at a high rating regardless of the rest of his game. The goal bonus is post-sigmoid
(`GOAL_ALPHA * goals^1.5`, capped at `GOAL_BONUS_CAP`) and is not scaled by minutes.

**Design (fixed before looking at results).**
- "Bad" is defined on **within-position non-scoring quality**, `non_scoring_dot = dot - scoring_dot` (the dot product minus the
  goals/assists contribution), ranked into percentiles within each position, so the definition is independent of the goal being studied.
  The bottom 50% is the main cut; the bottom decile is reported alongside.
- **Headline:** among bottom-half performances that scored, the share rated 7.5 or more and the lowest rating received.
- **Clean comparison:** bottom-half performances that did not score; the rating gap between the two groups, holding non-scoring quality
  roughly fixed, is the rescue effect.
- Positions: ST, Winger, WM and CM (descending goal alpha). Single-position players, pooled across careers; careers are shown separately
  where they have enough rows.

In [1]:
from pathlib import Path
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns

project_root = Path("..").resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from workshop.lib import crossval, datasets, decompose, fit, gate, inference, rating, refit, sweep, variants
from workshop.lib.report import ci, header, say

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [2]:
info = header()

**Validated config: `005d563f`** | run 2026-10-25

- Config fitted on: arsenal_2 1-38, ipswich_3 1-22, valencia_cf_1 1-157
- Data analysed (matches per career): arsenal_2 (38), ipswich_3 (22), valencia_cf_1 (157)

In [3]:
RANDOM_SEED = 42
N_BOOTSTRAP_RESAMPLES = 5000
BAD_FRACTION = 0.50
VERY_BAD_FRACTION = 0.10
RESCUE_THRESHOLD = 7.5
MIN_ROWS = 10
POSITION_GROUP_MAP = {"ST": "ST", "CF": "ST", "LW": "Winger", "RW": "Winger", "LM": "WM", "RM": "WM", "CM": "CM"}
GROUPS = ["ST", "Winger", "WM", "CM"]

## Data

In [4]:
svc = rating.load_service()
dec = decompose.decompose_outfield("pool")
dec["group"] = dec["position"].map(POSITION_GROUP_MAP)
dec["non_scoring_dot"] = dec["dot"] - dec["scoring_dot"]
dec["event_bonus"] = dec["goal_bonus"] + dec["assist_bonus"]
dec = dec[dec["group"].isin(GROUPS)].copy()
dec["nsq_pct"] = dec.groupby("group")["non_scoring_dot"].rank(pct=True)
dec["scored"] = dec["goals"] >= 1
dec["bad"] = dec["nsq_pct"] <= BAD_FRACTION
dec["very_bad"] = dec["nsq_pct"] <= VERY_BAD_FRACTION

summary = dec.groupby("group").agg(
    rows=("rating", "size"), scorers=("scored", "sum"), bad_rows=("bad", "sum"),
    bad_scorers=("scored", lambda s: int((s & dec.loc[s.index, "bad"]).sum())),
)
display(summary.loc[GROUPS])
print(f"GOAL_BONUS_CAP = {svc.GOAL_BONUS_CAP}; GOAL_ALPHA = {dict(svc.GOAL_ALPHA)}")

,rows,scorers,bad_rows,bad_scorers
group,,,,
ST,322,141,161,35
Winger,546,143,273,50
WM,69,11,34,6
CM,536,106,268,30


GOAL_BONUS_CAP = 1.8; GOAL_ALPHA = {'ST': 0.3, 'RW': 0.45, 'LW': 0.45, 'RM': 0.35, 'LM': 0.35, 'CM': 0.25, 'CAM': 0.3, 'CDM': 0.2, 'CB': 0.2, 'RB': 0.2, 'LB': 0.2, 'RWB': 0.25, 'LWB': 0.25}


## Core cut: do goals rescue bad performances?

In [5]:
def rescue_table(mask_col: str, frame: pd.DataFrame = dec) -> pd.DataFrame:
    rows = []
    for group in GROUPS:
        sub = frame[(frame["group"] == group) & frame[mask_col]]
        scored, blank = sub[sub["scored"]], sub[~sub["scored"]]
        if len(scored) < MIN_ROWS or len(blank) < MIN_ROWS:
            rows.append({"group": group, "n_scored": len(scored), "n_not_scored": len(blank)})
            continue
        gap = inference.mean_gap_bootstrap(scored["rating"], blank["rating"], N_BOOTSTRAP_RESAMPLES, RANDOM_SEED)
        rows.append({
            "group": group, "n_scored": len(scored), "n_not_scored": len(blank),
            "mean_not_scored": blank["rating"].mean(), "mean_scored": scored["rating"].mean(),
            "rescue_gap": gap.estimate, "ci_low": gap.ci_low, "ci_high": gap.ci_high,
            "share_scored_ge_7_5": (scored["rating"] >= RESCUE_THRESHOLD).mean(),
            "lowest_scorer_rating": scored["rating"].min(),
        })
    columns = ["group", "n_scored", "n_not_scored", "mean_not_scored", "mean_scored", "rescue_gap", "ci_low",
               "ci_high", "share_scored_ge_7_5", "lowest_scorer_rating"]
    return pd.DataFrame(rows, columns=columns).set_index("group")


print("BOTTOM HALF on non-scoring quality")
core = rescue_table("bad")
display(core)
print("BOTTOM DECILE on non-scoring quality")
decile = rescue_table("very_bad")
decile

BOTTOM HALF on non-scoring quality


,n_scored,n_not_scored,mean_not_scored,mean_scored,rescue_gap,ci_low,ci_high,share_scored_ge_7_5,lowest_scorer_rating
group,,,,,,,,,
ST,35,126,5.696,6.251,0.555,0.450,0.663,0.000,5.800
Winger,50,223,5.655,6.398,0.743,0.587,0.912,0.080,5.600
WM,6,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CM,30,238,5.741,6.313,0.573,0.445,0.705,0.000,5.600


BOTTOM DECILE on non-scoring quality


,n_scored,n_not_scored,mean_not_scored,mean_scored,rescue_gap,ci_low,ci_high,share_scored_ge_7_5,lowest_scorer_rating
group,,,,,,,,,
ST,1,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Winger,7,47,NaN,NaN,NaN,NaN,NaN,NaN,NaN
WM,1,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CM,3,50,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
tested = core.dropna(subset=["rescue_gap"])
skipped = core.index[core["rescue_gap"].isna()].tolist()
say(
    "**Bottom half of non-scoring quality: scored vs not scored.** "
    + "; ".join(
        f"{g}: {r['mean_scored']:.2f} vs {r['mean_not_scored']:.2f}, gap {r['rescue_gap']:+.2f} [{r['ci_low']:+.2f}, {r['ci_high']:+.2f}] "
        f"({int(r['n_scored'])} scorers; {r['share_scored_ge_7_5']:.0%} rated 7.5+, lowest {r['lowest_scorer_rating']:.1f})"
        for g, r in tested.iterrows()
    )
    + (f". Too few rows to compare: {', '.join(skipped)} ({', '.join(str(int(core.loc[s, 'n_scored'])) for s in skipped)} scorers)." if skipped else ".")
    + f" Bottom decile: " + ("not estimated for " + ", ".join(decile.index[decile['rescue_gap'].isna()]) if decile['rescue_gap'].isna().any() else "estimated for all")
    + ("; for the rest the gap is " + ", ".join(f"{g} {r['rescue_gap']:+.2f}" for g, r in decile.dropna(subset=['rescue_gap']).iterrows()) if decile['rescue_gap'].notna().any() else "")
    + "."
)

**Bottom half of non-scoring quality: scored vs not scored.** ST: 6.25 vs 5.70, gap +0.56 [+0.45, +0.66] (35 scorers; 0% rated 7.5+, lowest 5.8); Winger: 6.40 vs 5.66, gap +0.74 [+0.59, +0.91] (50 scorers; 8% rated 7.5+, lowest 5.6); CM: 6.31 vs 5.74, gap +0.57 [+0.45, +0.70] (30 scorers; 0% rated 7.5+, lowest 5.6). Too few rows to compare: WM (6 scorers). Bottom decile: not estimated for ST, Winger, WM, CM.

## Floor: the lowest rating any goal-scoring performance received

Across every goal-scoring performance (not only the bad ones), pooled and by career.

In [7]:
floor = (dec[dec["scored"]].groupby("group")["rating"]
         .agg(n="size", lowest="min", p10=lambda x: x.quantile(0.10), median="median"))
display(floor.loc[[g for g in GROUPS if g in floor.index]])
by_career = (dec[dec["scored"]].groupby(["group", "career"])["rating"].agg(n="size", lowest="min", median="median")
             .loc[lambda d: d["n"] >= MIN_ROWS])
by_career

,n,lowest,p10,median
group,,,,
ST,141,5.800,6.200,7.300
Winger,143,5.600,6.020,7.100
WM,11,6.000,6.000,6.600
CM,106,5.600,6.200,7.250


n  lowest  median
group  career                            
CM     arsenal_2       15   6.100   7.500
       valencia_cf_1   91   5.600   7.200
ST     arsenal_2       30   6.300   8.000
       ipswich_3       15   6.200   8.000
       valencia_cf_1   96   5.800   7.100
Winger arsenal_2       36   6.000   7.700
       valencia_cf_1  107   5.600   6.900

In [8]:
say(
    "**Lowest goal-scoring rating** by position: " + ", ".join(f"{g} {r['lowest']:.1f} (10th percentile {r['p10']:.2f}, median {r['median']:.2f})" for g, r in floor.loc[[g for g in GROUPS if g in floor.index]].iterrows())
    + ". Medians by career where there are enough scorers: "
    + ("; ".join(f"{g} " + ", ".join(f"{c} {by_career.loc[(g, c), 'median']:.2f}" for c in by_career.loc[g].index) for g in by_career.index.get_level_values(0).unique()) if len(by_career) else "none")
    + "."
)

**Lowest goal-scoring rating** by position: ST 5.8 (10th percentile 6.20, median 7.30), Winger 5.6 (10th percentile 6.02, median 7.10), WM 6.0 (10th percentile 6.00, median 6.60), CM 5.6 (10th percentile 6.20, median 7.25). Medians by career where there are enough scorers: CM arsenal_2 7.50, valencia_cf_1 7.20; ST arsenal_2 8.00, ipswich_3 8.00, valencia_cf_1 7.10; Winger arsenal_2 7.70, valencia_cf_1 6.90.

## By goal count and by minutes

The goal bonus takes no minutes, so a 20-minute cameo should earn the same bonus as a full game; this checks it in the data (goal bonus for a
single goal by minutes band) alongside how ratings grow with the number of goals.

In [9]:
scorers = dec[dec["scored"]].copy()
scorers["goal_count"] = scorers["goals"].clip(upper=3).astype(int).map({1: "1", 2: "2", 3: "3+"})
scorers["minutes_band"] = pd.cut(scorers["minutes"], [0, 29, 59, 90], labels=["<30", "30-59", "60+"])
by_goals = scorers[scorers["group"].isin(["ST", "Winger"])].pivot_table(index="group", columns="goal_count", values="rating", aggfunc=["mean", "count"])
display(by_goals)
one_goal = scorers[scorers["goals"] == 1]
by_minutes = one_goal.pivot_table(index="group", columns="minutes_band", values="goal_bonus", aggfunc="mean", observed=True)
counts_min = one_goal.pivot_table(index="group", columns="minutes_band", values="goal_bonus", aggfunc="count", observed=True)
display(by_minutes.loc[[g for g in GROUPS if g in by_minutes.index]])
counts_min.loc[[g for g in GROUPS if g in counts_min.index]]

mean             count        
goal_count     1     2    3+     1   2  3+
group                                     
ST         6.995 7.856 9.131    93  32  16
Winger     6.882 8.030 8.543   102  27  14

minutes_band,<30,30-59,60+
group,,,
ST,0.300,0.300,0.300
Winger,0.450,0.449,0.439
WM,NaN,NaN,0.350
CM,0.250,0.250,0.245


minutes_band,<30,30-59,60+
group,,,
ST,13.000,13.000,42.000
Winger,14.000,7.000,35.000
WM,NaN,NaN,3.000
CM,10.000,12.000,29.000


In [10]:
spread = (by_minutes.max(axis=1) - by_minutes.min(axis=1)).dropna()
say(
    "**Goals.** Mean rating by goal count (1 / 2 / 3+): "
    + "; ".join(f"{g} " + " / ".join(f"{by_goals.loc[g, ('mean', k)]:.2f}" for k in ["1", "2", "3"] if ('mean', k) in by_goals.columns and pd.notna(by_goals.loc[g, ('mean', k)])) for g in by_goals.index)
    + f". **Minutes.** The goal bonus for a single goal varies by at most {spread.max():.3f} between minutes bands "
    + ("(" + ", ".join(f"{g} {v:.3f}" for g, v in spread.items()) + "), so a cameo goal earns the same bonus as a full-game one." if spread.max() < 0.02
       else "(" + ", ".join(f"{g} {v:.3f}" for g, v in spread.items()) + "), so the bonus does depend on minutes in the data.")
)

**Goals.** Mean rating by goal count (1 / 2 / 3+): ST 6.99 / 7.86; Winger 6.88 / 8.03. **Minutes.** The goal bonus for a single goal varies by at most 0.011 between minutes bands (CM 0.005, ST 0.000, WM 0.000, Winger 0.011), so a cameo goal earns the same bonus as a full-game one.

## Decomposing the lift (bottom-half scorers vs bottom-half non-scorers)

Every term is in rating points, so the rescue gap splits into the goal bonus, the assist bonus, other bonuses, the base rating (which includes the
goal weighting inside the dot product) and the supremacy deduction.

In [11]:
PARTS = ["rating", "base_rating", "goal_bonus", "assist_bonus", "other_bonus", "supremacy_deduction"]
rows = []
for group in GROUPS:
    sub = dec[(dec["group"] == group) & dec["bad"]]
    for label, part in (("scored", sub[sub["scored"]]), ("not scored", sub[~sub["scored"]])):
        if len(part) >= MIN_ROWS:
            rows.append({"group": group, "bad_half": label, "n": len(part), **part[PARTS].mean().to_dict()})
decomp = pd.DataFrame(rows).set_index(["group", "bad_half"])
display(decomp)
both = [g for g in GROUPS if (g, "scored") in decomp.index and (g, "not scored") in decomp.index]
diff = pd.DataFrame({g: decomp.loc[(g, "scored"), PARTS] - decomp.loc[(g, "not scored"), PARTS] for g in both}).T
diff.columns = [f"d_{c}" for c in PARTS]
diff

n  rating  base_rating  goal_bonus  assist_bonus  other_bonus  supremacy_deduction
group  bad_half                                                                                        
ST     scored       35   6.251        5.948       0.378         0.000        0.052                0.131
       not scored  126   5.696        5.767       0.000         0.037        0.022                0.133
Winger scored       50   6.398        5.872       0.599         0.067        0.000                0.137
       not scored  223   5.655        5.729       0.000         0.072       -0.012                0.131
WM     not scored   28   5.779        5.755       0.000         0.032        0.007                0.009
CM     scored       30   6.313        6.018       0.291         0.075        0.073                0.139
       not scored  238   5.741        5.759       0.000         0.054        0.069                0.141

,d_rating,d_base_rating,d_goal_bonus,d_assist_bonus,d_other_bonus,d_supremacy_deduction
ST,0.555,0.181,0.378,-0.037,0.031,-0.001
Winger,0.743,0.143,0.599,-0.005,0.012,0.006
CM,0.573,0.260,0.291,0.021,0.004,-0.002


In [12]:
say(
    "**Where the lift comes from** (scored minus not scored, rating points): "
    + "; ".join(
        f"{g} {r['d_rating']:+.2f} = goal bonus {r['d_goal_bonus']:+.2f}, base rating {r['d_base_rating']:+.2f}, "
        f"assist bonus {r['d_assist_bonus']:+.2f}, other {r['d_other_bonus']:+.2f}"
        for g, r in diff.iterrows()
    ) + "."
)

**Where the lift comes from** (scored minus not scored, rating points): ST +0.56 = goal bonus +0.38, base rating +0.18, assist bonus -0.04, other +0.03; Winger +0.74 = goal bonus +0.60, base rating +0.14, assist bonus -0.00, other +0.01; CM +0.57 = goal bonus +0.29, base rating +0.26, assist bonus +0.02, other +0.00.

## Is it the same in every career?

Bottom-half rescue gap by career (cells need `MIN_ROWS` scorers and non-scorers).

In [13]:
rows = []
for career, part in dec.groupby("career"):
    for group in GROUPS:
        sub = part[(part["group"] == group) & part["bad"]]
        scored, blank = sub[sub["scored"]], sub[~sub["scored"]]
        if len(scored) < MIN_ROWS or len(blank) < MIN_ROWS:
            continue
        gap = inference.mean_gap_bootstrap(scored["rating"], blank["rating"], N_BOOTSTRAP_RESAMPLES, RANDOM_SEED)
        rows.append({"career": career, "group": group, "n_scored": len(scored), "n_not_scored": len(blank),
                     "rescue_gap": gap.estimate, "ci_low": gap.ci_low, "ci_high": gap.ci_high,
                     "share_scored_ge_7_5": (scored["rating"] >= RESCUE_THRESHOLD).mean()})
career_gaps = pd.DataFrame(rows)
career_gaps.set_index(["career", "group"]) if len(career_gaps) else career_gaps

n_scored  n_not_scored  rescue_gap  ci_low  ci_high  share_scored_ge_7_5
career        group                                                                           
valencia_cf_1 ST            31           106       0.556   0.442    0.674                0.000
              Winger        45           206       0.764   0.590    0.947                0.089
              CM            28           202       0.605   0.469    0.746                0.000

In [14]:
if len(career_gaps):
    say("**By career.** " + "; ".join(f"{r['career']} {r['group']} {r['rescue_gap']:+.2f} [{r['ci_low']:+.2f}, {r['ci_high']:+.2f}]" for _, r in career_gaps.iterrows())
        + ". Cells with fewer than the minimum scorers or non-scorers are skipped.")
else:
    say("No career has enough bottom-half scorers and non-scorers in any position for a separate estimate.")

**By career.** valencia_cf_1 ST +0.56 [+0.44, +0.67]; valencia_cf_1 Winger +0.76 [+0.59, +0.95]; valencia_cf_1 CM +0.61 [+0.47, +0.75]. Cells with fewer than the minimum scorers or non-scorers are skipped.

## Notes and limitations

- Classification is within position and pooled across careers, so a career's bottom half is judged against the pooled distribution.
- Bottom-half scorers are a small slice of each group; cells under `MIN_ROWS` are skipped, not estimated.
- No counterfactual re-rating: the rescue gap compares different performances at similar non-scoring quality; it does not re-rate the same
  performance without its goal.
- The decomposition is exact but the attribution is not causal: the base rating still contains the goals/assists weighting inside the dot product.

In [15]:
worst = tested["share_scored_ge_7_5"].max() if len(tested) else float("nan")
say(
    f"**Summary (config `{info['hash'][:8]}`).** Among bottom-half performances a goal lifts the rating by "
    + ", ".join(f"{g} {r['rescue_gap']:+.2f}" for g, r in tested.iterrows())
    + f"; at most {worst:.0%} of bad-but-scored performances in any position reach {RESCUE_THRESHOLD}, and the lowest goal-scoring ratings are "
    + ", ".join(f"{g} {floor.loc[g, 'lowest']:.1f}" for g in GROUPS if g in floor.index) + "."
)

**Summary (config `005d563f`).** Among bottom-half performances a goal lifts the rating by ST +0.56, Winger +0.74, CM +0.57; at most 8% of bad-but-scored performances in any position reach 7.5, and the lowest goal-scoring ratings are ST 5.8, Winger 5.6, WM 6.0, CM 5.6.